<a href="https://colab.research.google.com/github/pedrohue/pedrohue-ERP-requisitos-dos-clientes/blob/main/PROJETO_CODIGO_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import sqlite3

def configurar_banco():
  conexao = sqlite3.connect('projeto_estoque.db')
  cursor = conexao.cursor()

#criando a tabela de produtos
  cursor.execute("""
    CREATE TABLE IF NOT EXISTS produtos (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nome TEXT,
        categoria TEXT,
        preco REAL,
        quantidade INTEGER,
        especificacoes TEXT
    )
    """)

  cursor.execute ("""
    CREATE TABLE IF NOT EXISTS movimentacoes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        id_produto INTEGER,
        tipo TEXT,
        quantidade INTEGER
    )
    """)
  conexao.commit()
  conexao.close()

#executa a configuraçao inicial do banco
configurar_banco()

In [9]:
import sqlite3

def cadastrar_produto(nome,categoria,preco,quantidade,especificacoes):
  conexao = sqlite3.connect('projeto_estoque.db')
  cursor = conexao.cursor()

  #adiciona o novo produto
  cursor.execute("""
        INSERT INTO produtos (nome, categoria, preco, quantidade, especificacoes)
        VALUES (?, ?, ?, ?, ?)
    """, (nome, categoria, preco, quantidade, especificacoes))

  conexao.commit()
  conexao.close()
  print('produto cadastrado com sucesso!')


  def registrar_movimentacao(id_produto, tipo, quantidade):
    conexao = sqlite3.connect('projeto_estoque.db')
    cursor = conexao.cursor()

#busca a quantidade do produto
    cursor.execute("SELECT quantidade FROM produtos WHERE id = ?", (id_produto,))
    resultado = cursor.fetchone()

    if not resultado:
      print("Produto não encontrado.")
      conexao.close()
      return

    qtd_atual = resultado[0]

    #verifica se tem estoque suficiente
    if tipo == 'saida' and qtd_atual < quantidade:
        print("Quantidade insuficiente em estoque.")
        conexao.close()
        return

    if tipo == "ENTRADA":
        nova_qtd = qtd_atual + quantidade
    else:
        nova_qtd = qtd_atual - quantidade

    # Atualiza o saldo atual na tabela de produtos
    cursor.execute("UPDATE produtos SET quantidade = ? WHERE id = ?", (nova_qtd, id_produto))

    # Grava o histórico na tabela de movimentações
    cursor.execute("INSERT INTO movimentacoes (id_produto, tipo, quantidade) VALUES (?, ?, ?)",
                   (id_produto, tipo, quantidade))

    conexao.commit()
    conexao.close()
    print('movimentaçao registrada!')

In [10]:
#importaçoes

#importa o sqlite(banco de dados)
import sqlite3

#criar banco de dados
conn = sqlite3.connect('estoque.db', check_same_thread=False)
cursor = conn.cursor()

#tabela de usuarios
cursor.execute('''
create table if not exists usuarios (
    id integer primary key autoincrement,
    nome TEXT,
    email TEXT,
    senha TEXT
)
''')

#tabela de produtos
cursor.execute('''
create table if not exists produtos (
    id integer primary key autoincrement,
    nome TEXT,
    quantidade INTEGER,
    usuario_id INTEGER
)
''')

#salva as tabelas no banco de dados
conn.commit()


In [11]:
%%writefile Cadastro_produtos.py

import sqlite3

def conectar_banco():
    return sqlite3.connect("estoque.db")

def criar_tabela():
    conexao = conectar_banco()
    cursor = conexao.cursor()
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS estoque (
        codigo TEXT PRIMARY KEY,
        nome_item TEXT,
        quantidade REAL,
        unidade_medida TEXT,
        preco_unitario REAL,
        preco_total REAL
    )
    ''')
    conexao.commit()
    conexao.close()

def cadastrar_produto():
    print("\n--- CADASTRO DE PRODUTO ---")

    codigo = input("Digite o código do produto (001 a 999): ")

    if not codigo.isdigit() or len(codigo) != 3:
        print("Erro: o código deve conter 3 números. Exemplo: 001.")
        return

    nome_item = input("Digite o nome do item: ")
    quantidade = float(input("Digite a quantidade em estoque: "))
    unidade_medida = input("Digite a unidade de medida (un, kg, L, m): ")
    preco_unitario = float(input("Digite o preço unitário: "))

    preco_total = quantidade * preco_unitario

    produto = {
        "codigo": codigo,
        "nome_item": nome_item,
        "quantidade": quantidade,
        "unidade_medida": unidade_medida,
        "preco_unitario": preco_unitario,
        "preco_total": preco_total
    }

    conexao = conectar_banco()
    cursor = conexao.cursor()

    try:
        cursor.execute("""
        INSERT INTO estoque (
            codigo,
            nome_item,
            quantidade,
            unidade_medida,
            preco_unitario,
            preco_total
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """, (
            produto["codigo"],
            produto["nome_item"],
            produto["quantidade"],
            produto["unidade_medida"],
            produto["preco_unitario"],
            produto["preco_total"]
        ))

        conexao.commit()
        print("\nProduto cadastrado com sucesso!")
        print(f"Preço total em estoque: R$ {preco_total:.2f}")

    except sqlite3.IntegrityError:
        print("\nErro: já existe um produto com esse código.")

    conexao.close()


def listar_produtos():
    conexao = conectar_banco()
    cursor = conexao.cursor()

    cursor.execute("SELECT * FROM estoque")
    produtos = cursor.fetchall()

    print("\n--- PRODUTOS CADASTRADOS ---")

    if len(produtos) == 0:
        print("Nenhum produto cadastrado.")
    else:
        for produto in produtos:
            print(f"""
Código: {produto[0]}
Nome: {produto[1]}
Quantidade: {produto[2]}
Unidade: {produto[3]}
Preço unitário: R$ {produto[4]:.2f}
Preço total: R$ {produto[5]:.2f}
------------------------------
""")
            if produto[2] < 5:
                print(f"ALERTA: O produto '{produto[1]}' está com {produto[2]} unidades (Faltando unidades!)\n")

    conexao.close()


def menu():
    criar_tabela()

    while True:
        print("""
===== SISTEMA DE ESTOQUE =====
1 - Cadastrar produto
2 - Listar produtos
3 - Sair
""")

        opcao = input("Escolha uma opção: ")

        if opcao == "1":
            cadastrar_produto()
        elif opcao == "2":
            listar_produtos()
        elif opcao == "3":
            print("Sistema encerrado.")
            break
        else:
            print("Opção inválida. Tente novamente.")


menu()

Overwriting Cadastro_produtos.py


In [ ]:
!python Cadastro_produtos.py


===== SISTEMA DE ESTOQUE =====
1 - Cadastrar produto
2 - Listar produtos
3 - Sair

Escolha uma opção: 1

--- CADASTRO DE PRODUTO ---
Digite o código do produto (001 a 999): 001
Digite o nome do item: pc
Digite a quantidade em estoque: 2
Digite a unidade de medida (un, kg, L, m): un
Digite o preço unitário: 300

Produto cadastrado com sucesso!
Preço total em estoque: R$ 600.00

===== SISTEMA DE ESTOQUE =====
1 - Cadastrar produto
2 - Listar produtos
3 - Sair

Escolha uma opção: 2

--- PRODUTOS CADASTRADOS ---

Código: 001
Nome: pc
Quantidade: 2.0
Unidade: un
Preço unitário: R$ 300.00
Preço total: R$ 600.00
------------------------------

ALERTA: O produto 'pc' está com 2.0 unidades (Faltando unidades!)


===== SISTEMA DE ESTOQUE =====
1 - Cadastrar produto
2 - Listar produtos
3 - Sair

Escolha uma opção: 